In [1]:
!pip install -q -U bitsandbytes 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00


In [ ]:
import time
import torch
import random
import re
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ===========================  1. 模型切换开关 ===========================
# 先保持 Qwen 跑一次；跑完后重启 Kernel，注释掉 Qwen，取消注释 Llama 再跑一次。
# CURRENT_MODEL = "Qwen/Qwen2.5-7B-Instruct" 
CURRENT_MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit" 

SEED = 42
# ==============================================================================

print(f"🤖 正在初始化模型 (无 CoT 模式): {CURRENT_MODEL} ...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(CURRENT_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    CURRENT_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
if tokenizer.pad_token_id is None: tokenizer.pad_token_id = tokenizer.eos_token_id

# 获取大致参数量 (Million)
MODEL_PARAMS_M = sum(p.numel() for p in model.parameters()) / 1e6

# --- 2. 标签映射与兜底策略 ---
LABEL_MAPPING = {
    "SMP2020": {"fallback": 3, "max_id": 5},
    "SST-5": {"fallback": 2, "max_id": 4},
    "TweetEval": {"fallback": 1, "max_id": 2}
}

# --- 3. 验证集加载 (100% 复刻小模型的验证集逻辑) ---
def get_validation_set(dataset_name):
    # 强制固定种子，保证与小模型切出的验证集完全一致
    seed_val = 42 
    np.random.seed(seed_val)
    random.seed(seed_val)
    
    if dataset_name == "SMP2020":
        ds = load_dataset("Um1neko/smp2020", split="train")
        df = pd.DataFrame(ds).rename(columns={"content": "text"})
        val_count = 50  # 与 HiPro SMP2020 对齐
    elif dataset_name == "SST-5":
        ds = load_dataset("SetFit/sst5", split="train")
        df = pd.DataFrame(ds).rename(columns={"sentence": "text"})
        val_count = 80  # 与 HiPro SST-5 对齐
    elif dataset_name == "TweetEval":
        ds = load_dataset("tweet_eval", "sentiment", split="train")
        df = pd.DataFrame(ds)
        val_count = 50  # 与 HiPro TweetEval 对齐
        
    df = df[["text", "label"]].dropna()
    df["label"] = df["label"].astype(int)
    
    # 直接拿全量数据按 42 切分。Prompt里的14个例子本身就在train_pool里。
    # 这样切出来的 val_pool 和小模型代码一模一样！
    _, val_pool = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=seed_val)
    
    num_labels = df['label'].nunique()
    sampled_dfs = []
    for label in range(num_labels):
        class_df = val_pool[val_pool['label'] == label]
        n_samples = min(len(class_df), val_count)
        sampled_dfs.append(class_df.sample(n=n_samples, random_state=seed_val))
    
    val_df = pd.concat(sampled_dfs).sample(frac=1, random_state=seed_val).reset_index(drop=True)
    return val_df

# --- 4. 纯净版 Prompt 构造器 (无 Reasoning) ---
def build_prompt(dataset_name, text, method):
    
    if dataset_name == "SMP2020":
        task_desc = "任务：判断以下中文文本的情感类别。\n选项：0:愤怒, 1:悲伤, 2:恐惧, 3:中性, 4:高兴, 5:惊奇"
        examples = """
参考示例：
文本："谁能告诉我怎么把flyme账户这个SB关了" -> 答案：0
文本："明明想做你最终的伴侣却又无能为力……" -> 答案：1
文本："发现里面有个人，定睛一看，原来是我妹妹，吓得我一愣一愣的，到现在还心有余悸！" -> 答案：2
文本："那份内部报告一经泄露，引发了一场前所未有的公共信任危机。" -> 答案：3
文本："紧张那么些天终于过了第一次窄角停车了，好开心加油" -> 答案：4
文本："想不到！河海大学图书馆借阅次数最高的居然是····高数！" -> 答案：5
"""
        target_block = f"请仅输出一个数字 ID (0-5)，不要输出任何解释。\n文本：\"{text}\"\n答案："

    elif dataset_name == "SST-5":
        task_desc = "Task: Classify the sentiment of the text.\nOptions: 0:Very Negative, 1:Negative, 2:Neutral, 3:Positive, 4:Very Positive"
        examples = """
Examples:
Text: "an ugly , pointless , stupid movie ." -> Answer: 0
Text: "... the efforts of its star , kline , to lend some dignity to a dumb story are for naught ." -> Answer: 1
Text: "avary 's film never quite emerges from the shadow of ellis ' book ." -> Answer: 2
Text: "the film does give a pretty good overall picture of the situation in laramie following the murder of matthew shepard ." -> Answer: 3
Text: "... a powerful sequel and one of the best films of the year ." -> Answer: 4
"""
        target_block = f"Return ONLY the numeric ID (0-4). Do not explain.\nText: \"{text}\"\nAnswer:"

    elif dataset_name == "TweetEval":
        task_desc = "Task: Classify tweet sentiment.\nOptions: 0:Negative, 1:Neutral, 2:Positive"
        examples = """
Examples:
Text: "Everton Vs. Chelsea so early on a Saturday. These boys want to ruin my weekend. Why Lord." -> Answer: 0
Text: "JR’s HAVE to go to the pep rally tomorrow!" -> Answer: 1
Text: "Foo Fighters tomorrow night in Milton Keynes also supported by Royal Blood, can't wait, going to be awesome!" -> Answer: 2
"""
        target_block = f"Return ONLY the numeric ID (0-2). Do not explain.\nText: \"{text}\"\nAnswer:"

    # 根据方法决定是否加入 Examples
    if method == "Zero-Shot (No CoT)":
        return f"{task_desc}\n\n{target_block}"
    else: # Balanced 1-Shot (No CoT)
        return f"{task_desc}\n{examples}\n{target_block}"

# --- 5. 纯数字解析器 ---
def parse_prediction(response, dataset_name):
    mapping_info = LABEL_MAPPING[dataset_name]
    matches = re.findall(r'\d', response)
    if matches:
        pred = int(matches[0])
        if 0 <= pred <= mapping_info["max_id"]:
            return pred
    return mapping_info["fallback"]

# --- 6. 核心推理函数 (带性能监控 & Cases 保存) ---
def run_benchmark(dataset_name, method):
    print(f"\n🚀 {dataset_name} | {method} ...")
    val_df = get_validation_set(dataset_name)
    
    preds = []
    responses = []
    labels = val_df['label'].tolist()
    
    # 清理显存
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    sys_msg = "You are a sentiment classification system. You must output ONLY a single digit as the answer."

    start_time = time.time()

    for text in tqdm(val_df['text']):
        content = build_prompt(dataset_name, text, method)
        messages = [{"role": "system", "content": sys_msg}, {"role": "user", "content": content}]
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text_input], return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=5,     # ⚡ 物理切断解释能力，强迫只输出数字
                do_sample=False, 
                temperature=0.0,
                pad_token_id=tokenizer.pad_token_id
            )
        
        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        preds.append(parse_prediction(response, dataset_name))
        responses.append(response)
        
    end_time = time.time()
    
    # 物理性能计算
    total_time_sec = end_time - start_time
    avg_latency_ms = (total_time_sec / len(val_df)) * 1000 # 换算成毫秒
    peak_memory_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
    
    # 学术指标计算
    f1 = f1_score(labels, preds, average="macro")
    acc = accuracy_score(labels, preds)
    
    # 保存具体 Cases
    cases_df = pd.DataFrame({
        "text": val_df['text'],
        "true_label": labels,
        "pred_label": preds,
        "llm_response": responses # 无 CoT 虽然没有长篇大论，存下原始输出也很重要
    })
    safe_method_name = method.replace(" ", "_")
    safe_model_name = CURRENT_MODEL.split('/')[-1]
    cases_filename = f"{safe_model_name}_{dataset_name}_{safe_method_name}_cases.csv"
    cases_df.to_csv(cases_filename, index=False)
    
    return f1, acc, avg_latency_ms, peak_memory_mb

# --- 7. 启动双基线实验 ---
results = []
datasets = ["SMP2020", "SST-5", "TweetEval"]
methods = ["Zero-Shot (No CoT)", "Balanced 1-Shot (No CoT)"]

print(f"{'='*80}\n🏆 启动公平评估协议 (完全对齐版 | 无 CoT): {CURRENT_MODEL}\n{'='*80}")

for ds in datasets:
    for method in methods:
        f1, acc, latency, memory = run_benchmark(ds, method)
        print(f"✅ {ds} ({method}) -> F1: {f1:.4f}, Acc: {acc:.4f}, Latency: {latency:.2f}ms, VRAM: {memory:.2f}MB")
        
        results.append({
            "Dataset": ds, 
            "Model": f"{CURRENT_MODEL.split('/')[-1]}", 
            "Method": method,
            "Macro-F1": f1, 
            "Accuracy": acc,
            "Inference_Time_ms": latency,
            "Peak_Memory_MB": memory,
            "Params_M": MODEL_PARAMS_M
        })

final_df = pd.DataFrame(results)
print("\n" + "="*80 + "\n📊 最终评测报表 (无 CoT 组 | 全指标)\n" + "="*80)
print(final_df.to_string(index=False))

# 保存结果
save_name = f"{CURRENT_MODEL.split('/')[-1]}_aligned_NoCoT_results.csv"
final_df.to_csv(save_name, index=False)
print(f"\n💾 数据已保存至: {save_name}")

🤖 正在初始化模型 (无 CoT 模式): unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit ...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

  0%|          | 0/150 [00:00<?, ?it/s]

✅ TweetEval (Zero-Shot (No CoT)) -> F1: 0.3938, Acc: 0.4333, Latency: 587.56ms, VRAM: 5591.07MB

🚀 TweetEval | Balanced 1-Shot (No CoT) ...


  0%|          | 0/150 [00:00<?, ?it/s]

✅ TweetEval (Balanced 1-Shot (No CoT)) -> F1: 0.5516, Acc: 0.6000, Latency: 887.75ms, VRAM: 5607.97MB

📊 最终评测报表 (无 CoT 组 | 全指标)
  Dataset                               Model                   Method  Macro-F1  Accuracy  Inference_Time_ms  Peak_Memory_MB   Params_M
  SMP2020 Meta-Llama-3.1-8B-Instruct-bnb-4bit       Zero-Shot (No CoT)  0.457400  0.496667         707.076400     5614.976074 4540.60032
  SMP2020 Meta-Llama-3.1-8B-Instruct-bnb-4bit Balanced 1-Shot (No CoT)  0.540008  0.576667        1160.322696     5656.403320 4540.60032
    SST-5 Meta-Llama-3.1-8B-Instruct-bnb-4bit       Zero-Shot (No CoT)  0.423725  0.467500         631.764177     5598.456543 4540.60032
    SST-5 Meta-Llama-3.1-8B-Instruct-bnb-4bit Balanced 1-Shot (No CoT)  0.534590  0.550000         998.432533     5627.129395 4540.60032
TweetEval Meta-Llama-3.1-8B-Instruct-bnb-4bit       Zero-Shot (No CoT)  0.393809  0.433333         587.555526     5591.074219 4540.60032
TweetEval Meta-Llama-3.1-8B-Instruct-bnb-4bit Bala